# Checks of the solver and the dataset tools
Each cell tests one property and ends with an `assert`; run all cells to validate an installation. Tolerances are in the variables at the top of each cell.

In [1]:
import os, tempfile
import numpy as np
from fpde_solver import *

N, T, NT = 128, 1.0, 800
eq, lin = burgers(nu=0.05, c=1.0), burgers(nu=0.05, c=0.0)               # nonlinear and linear (c = 0)
u0 = eq.ic(4, N, 2, 0)

**1. Linear case against the exact solution** (Mittag-Leffler): each Fourier mode follows $E_\alpha(-\nu|k|^\beta t^\alpha)\hat u_k(0)$.

In [2]:
TOL = 1.0                                                                 # % relative L2 error at the final time
t = np.linspace(0, T, 5)
for a, b in ((1.0, 2.0), (1.0, 1.5), (0.6, 2.0), (0.6, 1.5)):
    num = solve(u0, a, b, T, NT, lin, save_every=NT // 4)[1]
    err = rel_l2(num[:, -1], exact_linear(u0, a, b, t, lin)[:, -1]).max()
    print(f'alpha={a}, beta={b}: max error {err:.3f}%')
    assert err < TOL

alpha=1.0, beta=2.0: max error 0.002%


alpha=1.0, beta=1.5: max error 0.001%


alpha=0.6, beta=2.0: max error 0.003%


alpha=0.6, beta=1.5: max error 0.003%


**2. Classical Burgers** ($\alpha=1,\beta=2$) against an independent integrating-factor RK4 solver.

In [3]:
TOL = 2.0
num = solve(u0, 1.0, 2.0, T, NT, eq, save_every=NT)[1][:, -1]
ref = burgers_ref(u0, 0.05, 1.0, T, 1)[:, -1]
err = rel_l2(num, ref).max()
print(f'max error {err:.3f}%')
assert err < TOL

max error 0.114%


**3. Convergence in the time step**: the error against a fine run halves when `Nt` doubles (first order in time). Small $\alpha$ and $\beta$ need more steps to be stable at all (check 7).

In [4]:
a, b, NT_REF, RATIO = 0.6, 2.0, 6400, (1.6, 2.6)                          # a case that is stable at the smallest Nt
ref = solve(u0, a, b, T, NT_REF, eq, save_every=NT_REF)[1][:, -1]
errs = [rel_l2(solve(u0, a, b, T, n, eq, save_every=n)[1][:, -1], ref).max() for n in (200, 400, 800)]
ratios = np.array(errs[:-1]) / np.array(errs[1:])
print('Nt = 200, 400, 800: errors', np.round(errs, 3), '%, ratios', np.round(ratios, 2), '(2 = first order)')
assert ((ratios > RATIO[0]) & (ratios < RATIO[1])).all()

Nt = 200, 400, 800: errors [0.166 0.08  0.038] %, ratios [2.06 2.13] (2 = first order)


**4. Conservation of the mean** (the $k=0$ mode is untouched by the dissipation and by the flux), for the three kinds of runs.

In [5]:
TOL = 1e-8
for a, b in ((1.0, 1.5), (0.6, 2.0), (0.6, 1.5)):
    u = solve(u0, a, b, T, 1600, eq, save_every=400)[1]
    drift = np.abs(u.mean(-1) - u.mean(-1)[:, :1]).max()
    print(f'alpha={a}, beta={b}: max drift of the mean {drift:.1e}')
    assert drift < TOL

alpha=1.0, beta=1.5: max drift of the mean 4.5e-17


alpha=0.6, beta=2.0: max drift of the mean 2.9e-17


alpha=0.6, beta=1.5: max drift of the mean 3.2e-17


**5. Dataset files**: shapes, train/test split, reproducibility, and `N_out` (solve fine, save coarse) equal to the spectral truncation of the fine solution.

In [6]:
N_FINE, N_OUT, N_SAMPLES, N_TEST, FRAMES = 128, 64, 6, 2, 11
args = (eq, 1.0, 1.5, N_SAMPLES, N_FINE, 2 * np.pi, T, 400, 400 // (FRAMES - 1), 2, 0)
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, 'd.npz')
    d = make_dataset(*args, path=path, N_out=N_OUT, n_test=N_TEST)
    u_train, u_test, t = load_dataset(path)
    assert u_train.shape == (N_SAMPLES - N_TEST, FRAMES, N_OUT) and u_test.shape == (N_TEST, FRAMES, N_OUT) and len(t) == FRAMES
    assert u_train.dtype == np.float32
    fine = make_dataset(*args)['u']
    trunc = np.fft.irfft(np.fft.rfft(fine, axis=-1)[..., :N_OUT // 2 + 1] * (N_OUT / N_FINE), n=N_OUT, axis=-1)
    assert np.allclose(np.concatenate([u_train, u_test]), trunc, atol=1e-5)
    assert np.allclose(make_dataset(*args)['u'], fine)                    # same seed, same data
print('shapes', u_train.shape, u_test.shape, '| coarse = truncated fine | reproducible: ok')

shapes (4, 11, 64) (2, 11, 64) | coarse = truncated fine | reproducible: ok


**6. Noise helper**: the noise std is `sigma` times the std of the snapshots at the same time; the clean array is untouched.

In [7]:
SIGMA, TOL = 0.1, 0.05
u = make_dataset(eq, 1.0, 1.5, 100, N, 2 * np.pi, T, 800, 80, 2, 0)['u']
v = add_noise(u, SIGMA, seed=1)
ratio = (v - u).std(axis=(0, 2)) / u.std(axis=(0, 2))
print('noise std / snapshot std at each saved time: min %.3f, max %.3f (target %.2f)' % (ratio.min(), ratio.max(), SIGMA))
assert np.abs(ratio - SIGMA).max() < TOL * SIGMA * 4 and v.shape == u.shape and not np.allclose(u, v)
assert np.allclose(add_noise(u, 0.0), u)

noise std / snapshot std at each saved time: min 0.098, max 0.102 (target 0.10)


**7. Stability check**: an easy case is stable at the smallest tested `Nt`, a hard case (small $\alpha$ and $\beta$) is reported as unstable.

In [8]:
easy, hard = min_stable_steps(eq, 1.0, 2.0, T), min_stable_steps(eq, 0.4, 1.2, T)
print('alpha=1, beta=2: smallest stable Nt =', easy, '| alpha=0.4, beta=1.2:', hard)
assert easy == 200 and hard is None

alpha=1, beta=2: smallest stable Nt = 200 | alpha=0.4, beta=1.2: None


All checks passed if every cell ran without an assertion error.